In [1]:
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the program, can I still enroll?"

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Valinor\Programs\miniconda3\envs\llm_zoomcamp\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shash\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
v1 = model.encode(q1)

In [4]:
v1.shape

(384,)

In [5]:
v2 = model.encode(q2)

In [6]:
q1 = 'Can I still join the course after the start date?'
v1 = model.encode(q1)

In [7]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [8]:
v1.dot(dv)

np.float32(0.32332397)

In [9]:
q2 = 'How to install Docker on Windows?'
v2 = model.encode(q2)

In [10]:
v2.dot(dv)

np.float32(0.019730467)

In [ ]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

'wget' is not recognized as an internal or external command,
operable program or batch file.


In [12]:
!curl -O https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100    888 100    888   0      0   2309      0                              0
100    888 100    888   0      0   2309      0                              0
100    888 100    888   0      0   2309      0                              0


In [13]:
from ingest import load_faq_data

documents = load_faq_data()

In [14]:
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

In [15]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [16]:
len(texts)

1350

In [17]:
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

In [18]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [19]:
scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

In [20]:
import numpy as np
X = np.array(vectors)

In [21]:
scores = X.dot(v1)

In [22]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.762941))

In [23]:
documents[553]

{'course': 'llm-zoomcamp',
 'section': 'Module 1: RAG',
 'question': 'OpenAI: Error when running OpenAI responses.create command',
 'answer': 'You may receive the following error when running the OpenAI `responses.create` command due to insufficient credits in your OpenAI account:\n\n```\nOpenAI API Error: Insufficient credits\n```',
 'doc_id': 'f5df151c59'}

In [24]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]

In [25]:
scores[top5]

array([0.762941  , 0.7579371 , 0.7192131 , 0.6536312 , 0.56009996],
      dtype=float32)

In [26]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579371
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 

In [27]:
top5

array([  2, 625, 907, 538,   7])

In [28]:
top5 = np.argsort(-scores)[:5]

In [29]:
top5

array([  2, 625, 907, 538,   7])

In [30]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [31]:
vindex.search(v1, num_results=5, filter_dict={'course': 'llm-zoomcamp'})

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.',
  'doc_id': 'bd31146b0e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed an

In [32]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

'wget' is not recognized as an internal or external command,
operable program or batch file.


In [33]:
!curl -O https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100   2134 100   2134   0      0   5682      0                              0
100   2134 100   2134   0      0   5681      0                              0
100   2134 100   2134   0      0   5679      0                              0


In [36]:
from dotenv import load_dotenv
from openai import OpenAI
import os
load_dotenv()

# uncomment below to use openai
# openai_client = OpenAI()

# uncomment below if using groq instead
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [35]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [39]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    model='qwen/qwen3-32b')   # if using openai instead of groq, no need to pass model parameter

In [40]:
query = 'I just found out about the program, can I still sign up?'
assistant.rag(query)

"**Answer:**  \nYes, you can still join the program even if you just discovered it. However, if you want to receive a certificate, you must submit your **Capstone project** before the submission deadline. Homework is optional but recommended for practice and leaderboard ranking. \n\nIf you're setting up for specific modules (e.g., Module 2 or dlt workshops), follow setup instructions in the relevant sections (e.g., using a new codespace for Module 2 or running commands like `dlt init filesystem duckdb`). Let the team know if you need further help!"

In [41]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [42]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
    model='qwen/qwen3-32b')   # if using openai instead of groq, no need to pass model parameter

In [44]:
query = 'I just found out about the program, can I still sign up?'
vector_assistant.rag(query)

'Yes, you can still join the program! Here’s what you need to know:\n\n1. **Registration is optional**: While registering helps the organizers gauge interest, it’s not required to access course materials or start learning. You can begin the course and submit homework right away, even without formal registration. The registration form remains open until the course start date.\n\n2. **Certificate eligibility**: To receive a certificate, you must **submit your project while the submission window is open**. If you register later, make sure to track the project submission deadlines (these are typically outlined in the course timeline). If you miss submission dates, you’ll still have full access to course content but won’t be eligible for a certificate.\n\n3. **Start learning immediately**: All course materials (like pre-recorded modules, code repositories, and homework) are publicly available in the course repo. You can explore them anytime.\n\n4. **For extra support (optional)**: If you wa

In [45]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

In [46]:
vs_index.fit(vectors, documents)

In [52]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [53]:
results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

In [54]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

In [55]:
vs_index.close()